In [ ]:
import matplotlib.pyplot as plt
import cobra

model=cobra.io.read_sbml_model("iML1515.xml")
carbon_sources = ["EX_glyc_e", "EX_gal_e", "EX_fru_e", "EX_ac_e", "EX_pyr_e", "EX_succ_e", "EX_xyl__D_e"]

yield_values = []
growth_values = []
labels = []

for carbon_rxn_id in carbon_sources:

    test_model = model.copy()

    for rxn_id in carbon_sources:
        if rxn_id in test_model.reactions:
            test_model.reactions.get_by_id(rxn_id).lower_bound = 0

    carbon_rxn = test_model.reactions.get_by_id(carbon_rxn_id)
    carbon_rxn.lower_bound = -10
    carbon_rxn.upper_bound = 1000

    test_model.reactions.get_by_id("EX_lac__D_e").lower_bound = 0

    test_model.objective = "BIOMASS_Ec_iML1515_core_75p37M"
    max_growth = test_model.optimize().objective_value

    test_model.reactions.get_by_id("BIOMASS_Ec_iML1515_core_75p37M").lower_bound = 0.5 * max_growth

    test_model.objective = "EX_lac__D_e"
    sol = test_model.optimize()

    l_flux = sol.fluxes["EX_lac__D_e"]
    substrate_uptake = abs(sol.fluxes[carbon_rxn_id])
    growth = sol.fluxes["BIOMASS_Ec_iML1515_core_75p37M"]

    yield_per_carbon = l_flux / substrate_uptake

    yield_values.append(yield_per_carbon)
    growth_values.append(growth)
    labels.append(carbon_rxn_id)

# Plot
plt.figure(figsize=(6,5))
plt.scatter(growth_values, yield_values)
for i, name in enumerate(labels):
    plt.text(growth_values[i], yield_values[i], name)
plt.xlabel("Growth rate")
plt.ylabel("lactate yield per substrate")
plt.title("Yield vs Growth for Different Carbon Sources")
plt.show()